# 26 — Pull ND-GAIN Country Index (Climate Vulnerability + Readiness)

**Source:** Notre Dame Global Adaptation Initiative (ND-GAIN) Country Index  
**Provider:** University of Notre Dame  
**Access:** Free direct CSV download — no registration required  
**Coverage:** 182 countries, 1995–present  
**Frequency:** Annual  

## What this notebook does
Downloads the ND-GAIN country index, which measures each country's **vulnerability**
to climate disruption and **readiness** to adapt. The key insight is the *interaction*:
climate stress affects instability risk differently depending on whether a country has
the institutional and economic capacity to respond.

## Why this matters for instability prediction
The climate-conflict literature consistently finds that drought/rainfall shocks raise
civil war onset risk, but the effect is moderated by adaptive capacity — functioning
water infrastructure, food reserves, and government effectiveness. The PRIO-GRID
features in the pipeline capture climate *exposure*; ND-GAIN adds the *adaptive
capacity* dimension. Most relevant for `civil_war_onset` in Sahel and Horn contexts.

## Features pulled

| Feature | Description |
|---|---|
| `ndgain_score` | Overall ND-GAIN score (0–100, higher = better) |
| `vulnerability` | Vulnerability component (0–1, higher = more vulnerable) |
| `readiness` | Readiness component (0–1, higher = more ready) |
| `vuln_food` | Food sector vulnerability |
| `vuln_water` | Water sector vulnerability |
| `vuln_health` | Health sector vulnerability |
| `vuln_ecosystem` | Ecosystem services vulnerability |
| `readiness_economic` | Economic readiness |
| `readiness_governance` | Governance readiness |
| `readiness_social` | Social readiness |

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
```

In [ ]:
import os
import io
import requests
import pandas as pd
from datetime import datetime
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

# ND-GAIN provides separate CSVs per component and one combined index file.
# The combined index file has overall score, vulnerability, and readiness.
# Sub-sector scores are in individual CSVs at the ND-GAIN GitHub mirror.
NDGAIN_INDEX_URL    = "https://gain.nd.edu/assets/521882/nd_gain_country_index_2024_release.csv"
NDGAIN_GITHUB_BASE  = "https://raw.githubusercontent.com/nyu-datascience/ND-GAIN-data/master"

# Sub-index component URLs on GitHub mirror
NDGAIN_COMPONENTS = {
    "vulnerability":          f"{NDGAIN_GITHUB_BASE}/vulnerability/vulnerability.csv",
    "readiness":              f"{NDGAIN_GITHUB_BASE}/readiness/readiness.csv",
    "vuln_food":              f"{NDGAIN_GITHUB_BASE}/vulnerability/sectors/food.csv",
    "vuln_water":             f"{NDGAIN_GITHUB_BASE}/vulnerability/sectors/water.csv",
    "vuln_health":            f"{NDGAIN_GITHUB_BASE}/vulnerability/sectors/health.csv",
    "vuln_ecosystem":         f"{NDGAIN_GITHUB_BASE}/vulnerability/sectors/ecosystem_services.csv",
    "readiness_economic":     f"{NDGAIN_GITHUB_BASE}/readiness/components/economic.csv",
    "readiness_governance":   f"{NDGAIN_GITHUB_BASE}/readiness/components/governance.csv",
    "readiness_social":       f"{NDGAIN_GITHUB_BASE}/readiness/components/social.csv",
}

print(f"ADLS target : abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/raw/ndgain/")
print(f"Run date    : {RUN_DATE}")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## Download combined ND-GAIN index

The main index file is in wide format: rows = countries, columns = years.
We melt to long (country × year) for the panel.

In [ ]:
def download_ndgain_csv(url: str, col_name: str) -> pd.DataFrame:
    """Download a wide-format ND-GAIN CSV and melt to (iso3, year, col_name)."""
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    df = pd.read_csv(io.StringIO(resp.text))
    # ND-GAIN CSVs: first column is ISO3 or country name, remaining columns are years
    id_col = df.columns[0]
    year_cols = [c for c in df.columns[1:] if str(c).isdigit()]
    df_long = df.melt(id_vars=[id_col], value_vars=year_cols,
                      var_name="year", value_name=col_name)
    df_long.rename(columns={id_col: "iso3"}, inplace=True)
    df_long["year"] = pd.to_numeric(df_long["year"], errors="coerce").astype("Int64")
    df_long[col_name] = pd.to_numeric(df_long[col_name], errors="coerce")
    return df_long[["iso3", "year", col_name]]


print("Downloading ND-GAIN combined index...", end=" ", flush=True)
resp_main = requests.get(NDGAIN_INDEX_URL, timeout=60)
resp_main.raise_for_status()
df_main = pd.read_csv(io.StringIO(resp_main.text))
print(f"done — shape: {df_main.shape}")
print(f"Columns: {list(df_main.columns[:10])}")
df_main.head(3)

## Parse and reshape main index

In [ ]:
# Main index is wide: ISO3 | Name | 1995 | 1996 | ... | 2022
# We detect the year columns and melt
id_cols_main = [c for c in df_main.columns if not str(c).isdigit()]
year_cols_main = [c for c in df_main.columns if str(c).isdigit()]

iso_col = [c for c in id_cols_main if "iso" in str(c).lower() or len(df_main[c].dropna().iloc[0]) == 3][0] \
    if id_cols_main else df_main.columns[0]

df_index_long = df_main.melt(id_vars=[iso_col], value_vars=year_cols_main,
                              var_name="year", value_name="ndgain_score")
df_index_long.rename(columns={iso_col: "iso3"}, inplace=True)
df_index_long["year"] = pd.to_numeric(df_index_long["year"]).astype("Int64")
df_index_long["ndgain_score"] = pd.to_numeric(df_index_long["ndgain_score"], errors="coerce")

print(f"Index long shape: {df_index_long.shape}")
print(f"Countries: {df_index_long['iso3'].nunique()} | Years: {df_index_long['year'].min()}–{df_index_long['year'].max()}")

## Download sub-index components

In [ ]:
from functools import reduce

component_frames = [df_index_long]

for comp_name, url in NDGAIN_COMPONENTS.items():
    try:
        df_comp = download_ndgain_csv(url, comp_name)
        n_obs = df_comp[comp_name].notna().sum()
        print(f"  {comp_name:30s}: {n_obs:,} non-null values")
        component_frames.append(df_comp)
    except Exception as e:
        print(f"  WARNING: could not download {comp_name}: {e}")

df_ndgain = reduce(lambda a, b: a.merge(b, on=["iso3", "year"], how="outer"), component_frames)
df_ndgain = df_ndgain.sort_values(["iso3", "year"]).reset_index(drop=True)

print(f"\nFinal shape: {df_ndgain.shape}")
print(f"Countries  : {df_ndgain['iso3'].nunique()}")
df_ndgain.head()

## Write to ADLS

In [ ]:
write_parquet(df_ndgain, f"raw/ndgain/{RUN_DATE}/ndgain_panel.parquet")

## Summary

In [ ]:
print("=" * 55)
print("ND-GAIN pull complete")
print("=" * 55)
print(f"  Rows (country-years) : {len(df_ndgain):,}")
print(f"  Countries            : {df_ndgain['iso3'].nunique()}")
print(f"  Year range           : {df_ndgain['year'].min()}–{df_ndgain['year'].max()}")
feat_cols = [c for c in df_ndgain.columns if c not in ("iso3", "year")]
print(f"  Features             : {feat_cols}")
print()
print("ADLS path written:")
print(f"  raw/ndgain/{RUN_DATE}/ndgain_panel.parquet")